> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAGzTIin49w/5zM403525G-teHXho4SLHg/view?utm_content=DAGzTIin49w&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h29b78664e1)。


# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install gradio==6.2.0 openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 beautifulsoup4==4.14.3 langchain_chroma==1.1.0

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 代码准备

由于存储到向量数据库前需要先准备好文档以及切分好的文档块，因此这里需要把上一节的内容进行载入：

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://zh.d2l.ai/chapter_introduction/index.html")
docs = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
 chunk_size = 1500,
 chunk_overlap = 150)
splits = text_splitter.split_documents(docs)
print(len(splits))

# 2. 向量数据库生成

## 2.1 简介

在将每一个文档切割成合适的 chunk 后，我们还需要进行文本嵌入及向量数据库存储：
- 文本嵌入：使用 Embedding 模型将文本转换成高维向量（如 1536 维）。
- 向量存储：将向量和元数据（metadata）一起存入向量数据库中。

存储完后，就相当于给文本建一个‘语义索引’，让模型能按‘意思相近’而不是‘关键词相同’来查资料。

## 2.2 Embedding 模型

Embeddings 的本质是将一段文本转化为一长串的向量，这些向量实际上是对文字的一种数字化表示。假如两段文本内容越相关，其在向量空间中的距离也是越近的。下面我们将使用 DashScopeEmbeddings 进行演示：

In [ ]:
from langchain_community.embeddings import DashScopeEmbeddings
import os

# 设置 embedding 模型（阿里云）
embeddings = DashScopeEmbeddings(
  dashscope_api_key=os.getenv('DASHSCOPE_API_KEY'))

# 设置文本内容
text_1 = "今天天气不错"

# 进行文本向量化
query_result = embeddings.embed_query(text_1)
print(query_result)

## 2.3 向量数据库存储

在准备好了 embedding 模型后，我们还需要解决的一个问题是用哪一个向量数据库进行向量的存储。

### 2.3.1 InMemoryVectorStore
在 LangChain 中最基础的向量数据库就是基于内存的 InMemoryVectorStore 。

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.embeddings import DashScopeEmbeddings
import os

vector_store = InMemoryVectorStore(embedding=DashScopeEmbeddings(
 dashscope_api_key=os.getenv('DASHSCOPE_API_KEY')))

我们可以通过 vectore_store.from_documents 将切分好的内容进行载入：

In [ ]:
vectordb = vector_store.from_documents(
  documents=splits,
  embedding=embeddings)

print(len(vectordb.store))

### 2.3.2 Chroma

但是 InMemoryVectorStore 无法进行长期保存，当程序运行结束后，向量数据库内的内容将自动清除。

因此为了能够更长久的保存，Langchain 提供了更专业的向量数据库支持，包括 Chroma、Pinecone 以及 FAISS 等。

那这里我就以 Chroma 为例来展示一下具体的使用方式（先安装相关库）：

In [ ]:
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_chroma import Chroma
import os

embeddings = DashScopeEmbeddings(
  dashscope_api_key=os.getenv('DASHSCOPE_API_KEY'))

vectordb = Chroma.from_documents(
  documents=splits,
  embedding=embeddings,
  persist_directory='./chroma')

print(vectordb._collection.count())